# Beatles Wordplay Pipeline
### Detecting cultural wordplay in scientific literature using GPT-4o-mini

End-to-end pipeline for the paper. Three stages:

1. **Stage 1a — Exact retrieval:** fetch article titles that match a Beatles song or
   lyric verbatim (with article/parenthetical variants).
2. **Stage 1b — Approximate retrieval:** fetch near-matches using relaxed
   leave-one-out queries (each non-article word dropped in turn) for wordplay candidates.
3. **Stage 2 — LLM classification:** label each candidate as genuine Beatles wordplay
   with GPT-4o-mini.

API keys are read from a local `.env` (see `.env.example`). Curated input lists live in
`data/`; retrieved data is written back to `data/`.

In [ ]:
import json
import re
import time
import unicodedata
from typing import List, Tuple

import pandas as pd
import requests
import urllib.parse
from tqdm.notebook import tqdm
from rapidfuzz import fuzz

import os
from dotenv import load_dotenv

load_dotenv()
SCOPUS_API_KEY = os.environ["SCOPUS_API_KEY"]

In [ ]:
BATCH_SIZE = 25       # results per Scopus page
MAX_RESULTS = 1_000   # cap per song for the approximate search

In [ ]:
def get_songs(fname: str) -> dict:
    """Read a 'name<TAB>year' list (with header) into {name: year}."""
    songs = {}
    with open(fname, "r", encoding="utf-8") as f_in:
        next(f_in)  # skip header
        for line in f_in:
            name, year = line.strip().split("\t")
            songs[name.strip('“”\"')] = int(year)
    return songs

In [ ]:
# Map non-ASCII characters to their closest ASCII equivalent.
replacements = {
    "æ": "ae", "ñ": "n", "ö": "oe", "ä": "ae", "ü": "ue", "ß": "ss",
    "ø": "o", "å": "a", "é": "e", "è": "e", "ç": "c", "ô": "o", "î": "i",
    "…": "...", "á": "a", "à": "a", "â": "a", "ã": "a", "ê": "e", "ë": "e",
    "í": "i", "ì": "i", "ï": "i", "ó": "o", "ò": "o", "õ": "o", "ú": "u",
    "ù": "u", "û": "u", "ý": "y", "ÿ": "y", "č": "c", "ž": "z", "š": "s",
    "ł": "l", "“": '\"', "”": '\"', "‘": "'", "’": "'", "–": "-", "—": "-",
    "·": "-", "°": " deg", "©": "", "®": "", "™": "", "µ": "u", "†": "*",
    "‡": "*", "×": "x"
}

In [ ]:
# Scopus JSON keys to extract, in output-column order.
fields = {
    "scopus_id": "dc:identifier",
    "title": "dc:title",
    "cited_by": "citedby-count",
    "author": "dc:creator",
    "venue": "prism:publicationName",
    "doi": "prism:doi",
    "cover_date": "prism:coverDisplayDate",
}

headers = {"Accept": "application/json", "X-ELS-APIKey": SCOPUS_API_KEY}

In [ ]:
def extract_year_safe(date_str: str):
    """Pull a 4-digit year from a Scopus cover-date string, or None."""
    match4 = re.search(r'\b(\d{2})\d{2}\b', date_str)
    if match4:
        return int(match4.group(0))

    match2 = re.search(r'(\d{2})\s*$', date_str)
    if match2:
        val = int(match2.group(1))
        if val <= 25:
            return 2000 + val
        elif 63 <= val <= 99:
            return 1900 + val
    return None

In [ ]:
def clean_line(line: str) -> str:
    """Normalise a string to plain ASCII for tab-separated output."""
    for src, tgt in replacements.items():
        line = line.replace(src, tgt)
    line = unicodedata.normalize("NFKD", line)
    return line.encode("ascii", "ignore").decode("ascii")

---
## Stage 1a — Exact reference retrieval

Fetch titles that match a song/lyric verbatim, allowing a missing leading article, a
parenthetical part, or removed periods. Output columns:
`row_nr, song_nr, song_name, cite_nr, scopus_id, title, author, cover_date, year, venue, cited_by, doi`.

In [ ]:
def build_url(song_name: str) -> Tuple[str, str, str]:
    """Exact-title query for one song, with article / parenthesis / period variants."""
    query = f'TITLE(\"{song_name}\")'

    for article in ["The ", "A "]:
        if song_name.startswith(article):
            query += f' OR TITLE(\"{song_name[len(article):]}\")'
            break

    if "(" in song_name and ")" in song_name:
        base = song_name.split("(")[0].strip()
        in_parens = song_name.split("(")[1][:-1].strip()
        query += f' OR TITLE(\"{base}\") OR TITLE(\"{in_parens}\")'

    if "." in song_name:
        query += f' OR TITLE(\"{song_name.replace(".", "")}\")'

    query_encoded = urllib.parse.quote(query, safe="")
    url = (
        "https://api.elsevier.com/content/search/scopus"
        f"?query={query_encoded}&count={BATCH_SIZE}"
    )
    return url, query, query_encoded

In [ ]:
def get_exact_refs(count: int, nr: int, song_name: str, song_year: int, fout: str) -> int:
    """Fetch exact matches for one song and append them to fout; returns running row count."""
    print(song_name)
    url, query, query_encoded = build_url(song_name)

    resp = requests.get(url, headers=headers)
    if resp.status_code != 200:
        print(f"Error {resp.status_code}: {resp.text}")
        return count

    data = json.loads(resp.text.encode("utf-8"))
    search_results = data.get("search-results")
    if search_results is None:
        print("No search results found.")
        return count

    total = search_results["opensearch:totalResults"]
    total = int(total) if total.isdigit() else 0

    cite_nr, batch_nr, wrote_results = 0, 0, False
    while batch_nr * BATCH_SIZE < total:
        for entry in data.get("search-results", {}).get("entry", []):
            scopus_id, title, cited_by, author, venue, doi, cover_date = (
                entry.get(key, "") for key in fields.values()
            )
            year = extract_year_safe(cover_date)
            if not isinstance(year, int) or year < song_year:
                continue

            title, author, venue = clean_line(title), clean_line(author), clean_line(venue)
            with open(fout, "a", encoding="utf-8") as f_out:
                f_out.write(
                    f"{count}\t{nr}\t{song_name}\t{cite_nr}\t{scopus_id}\t{title}\t{author}\t"
                    f"{cover_date}\t{year}\t{venue}\t{cited_by}\t{doi}\n"
                )
            wrote_results = True
            count += 1
            cite_nr += 1

        batch_nr += 1
        resp = requests.get(url + f"&start={batch_nr * BATCH_SIZE}", headers=headers)
        data = json.loads(resp.text.encode("utf-8"))

    if not wrote_results:  # placeholder row so every song stays represented
        with open(fout, "a", encoding="utf-8") as f_out:
            f_out.write(f"{nr}\t{song_name}\t-1\t\t\t\t\t\t\t\n")
    return count

In [ ]:
REF_HEADER = (
    "row_nr\tsong_nr\tsong_name\tcite_nr\tscopus_id\ttitle\tauthor\t"
    "cover_date\tyear\tvenue\tcited_by\tdoi\n"
)

# Exact matches for song titles.
fin, fout = "data/beatles_songs.txt", "data/scopus_song_refs.txt"
with open(fout, "w", encoding="utf-8") as f:
    f.write(REF_HEADER)

count = 0
for nr, (song, song_year) in enumerate(tqdm(get_songs(fin).items())):
    count = get_exact_refs(count, nr, song, song_year, fout)
    time.sleep(1)

In [ ]:
# Exact matches for lyric phrases.
fin, fout = "data/beatles_lyrics.txt", "data/scopus_lyric_refs.txt"
with open(fout, "w", encoding="utf-8") as f:
    f.write(REF_HEADER)

count = 0
for nr, (lyric, lyric_year) in enumerate(tqdm(get_songs(fin).items())):
    count = get_exact_refs(count, nr, lyric, lyric_year, fout)
    time.sleep(1)

---
## Stage 1b — Approximate (leave-one-out) retrieval

Generate candidates for wordplay that omits or changes one word: for every non-article
word, search for the title with that word dropped (excluding exact matches, already
captured in Stage 1a). Each hit gets a `rapidfuzz` similarity score against the song
title. Adds a `fuzz` column to the schema above.

In [ ]:
def build_urls(song_name: str) -> List[Tuple[str, str, str]]:
    """Leave-one-out queries: drop each non-article word in turn (titles of 3+ words)."""
    original = song_name.strip()
    for article in ["The ", "A ", "An "]:
        if original.startswith(article):
            song_name = original[len(article):]
            break
    else:
        song_name = original

    words = song_name.split()
    ignore_words = {"a", "an", "the"}
    if len(words) <= 2:
        return []

    urls = []
    original_phrase = " ".join(words)
    for i, word in enumerate(words):
        if word.lower() in ignore_words:
            continue
        partial1 = " ".join(words[:i])
        partial2 = " ".join(words[i + 1:])
        if not partial1:
            query = f'TITLE(\"{partial2}\")'
        elif not partial2:
            query = f'TITLE(\"{partial1}\")'
        else:
            query = f'TITLE(\"{partial1}\") AND TITLE(\"{partial2}\")'
        query += f' AND NOT TITLE(\"{original_phrase}\")'

        query_encoded = urllib.parse.quote(query, safe="")
        url = (
            "https://api.elsevier.com/content/search/scopus"
            f"?query={query_encoded}&count={BATCH_SIZE}"
        )
        urls.append((url, query, query_encoded))
    return urls

In [ ]:
def get_approx_refs(count: int, nr: int, song_name: str, song_year: int, fout: str) -> int:
    """Fetch leave-one-out matches for one song and append them to fout (with fuzz score)."""
    for url, query, query_encoded in build_urls(song_name):
        print(f"Query: {query}")
        cite_nr = 0

        resp = requests.get(url, headers=headers)
        if resp.status_code != 200:
            print(f"Error {resp.status_code}: {resp.text}")
            continue

        data = json.loads(resp.text.encode("utf-8"))
        search_results = data.get("search-results")
        if search_results is None:
            print("No search results found.")
            continue

        total = search_results["opensearch:totalResults"]
        total = int(total) if total.isdigit() else 0

        batch_nr = 0
        while batch_nr * BATCH_SIZE < total and cite_nr < MAX_RESULTS:
            for entry in data.get("search-results", {}).get("entry", []):
                scopus_id, title, cited_by, author, venue, doi, cover_date = (
                    entry.get(key, "") for key in fields.values()
                )
                year = extract_year_safe(cover_date)
                if not isinstance(year, int) or year < song_year:
                    continue

                title, author, venue = clean_line(title), clean_line(author), clean_line(venue)
                fuzz_score = fuzz.partial_ratio(title.lower(), song_name.lower())
                with open(fout, "a", encoding="utf-8") as f_out:
                    f_out.write(
                        f"{count}\t{nr}\t{song_name}\t{cite_nr}\t{scopus_id}\t{title}\t{author}\t"
                        f"{cover_date}\t{year}\t{venue}\t{cited_by}\t{doi}\t{fuzz_score}\n"
                    )
                count += 1
                cite_nr += 1

            batch_nr += 1
            resp = requests.get(url + f"&start={batch_nr * BATCH_SIZE}", headers=headers)
            data = json.loads(resp.text.encode("utf-8"))
    return count

In [ ]:
# Approximate (wordplay-candidate) matches for song titles.
fin, fout = "data/beatles_songs.txt", "data/scopus_approx_refs.txt"
with open(fout, "w", encoding="utf-8") as f:
    f.write(
        "row_nr\tsong_nr\tsong_name\tcite_nr\tscopus_id\ttitle\tauthor\t"
        "cover_date\tyear\tvenue\tcited_by\tdoi\tfuzz\n"
    )

count = 0
for nr, (song, song_year) in enumerate(tqdm(get_songs(fin).items())):
    print(f"Song: {song}")
    count = get_approx_refs(count, nr, song, song_year, fout)
    time.sleep(1)

---
## Stage 2 — LLM wordplay classification

Group the approximate candidates by song and ask GPT-4o-mini (deterministic,
`temperature = 0.0`) whether each title is genuine wordplay. Few-shot prompting with
human-annotated positive, negative, and hard-negative examples. Output columns:
`scopus_id, explanation, wordplay`.

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
MODEL = "gpt-4o-mini"
TEMPERATURE = 0.0

client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
# USD per 1M tokens for the model in use, for a cost estimate after the run.
token_costs = {
    "gpt-4o-mini": {"input": 0.40, "output": 1.60},
}

In [ ]:
def build_user_prompt(song_title: str, titles_to_classify: list[dict]) -> str:
    """Few-shot examples followed by the numbered list of titles to classify."""
    few_shot_examples = [
        {
            "beatles_song": "Let It Be",
            "paper_title": "There will be an answer; let it bleed",
            "explanation": "quotes the lyric and twists 'be' → 'bleed'",
            "answer": "Yes"
        },
        {
            "beatles_song": "With a Little Help from My Friends",
            "paper_title": "Understanding the role of friends in adolescent health decisions",
            "explanation": "topic overlap only, no playful re-phrasing",
            "answer": "No"
        },
        {
            "beatles_song": "Everybody's Got Something to Hide Except Me and My Monkey",
            "paper_title": ("Everybody's got something to hide except me and my patented monkey: "
                            "patentability of cloned organisms."),
            "explanation": "direct reuse of the song title with ironic twist",
            "answer": "Yes"
        },
        {
            "beatles_song": "The Long and Winding Road",
            "paper_title": "Myocyte hypertrophy: The long and winding RhoA'd",
            "explanation": "puns on 'Road' → 'RhoA'd'",
            "answer": "Yes"
        },
        {
            "beatles_song": "All You Need Is Love",
            "paper_title": ("All You Need Is RAW: Defending Against Adversarial Attacks "
                            "with Camera Image Pipelines"),
            "explanation": "swaps 'Love' for 'RAW' in the phrase",
            "answer": "Yes"
        },
        {
            "beatles_song": "Come Together",
            "paper_title": ("Come for climate, stay for community: Acting, emoting, and staying "
                            "together through the climate crisis"),
            "explanation": "contains all the same words but not used as a song reference",
            "answer": "No"
        },
    ]

    example_blocks = []
    for ex in few_shot_examples:
        block = (
            f"Example – **Song:** {ex['beatles_song']}\n"
            f"**Title:** \"{ex['paper_title']}\"\n"
            f"Assistant: {ex['answer']} – {ex['explanation']}"
        )
        example_blocks.append(block)
    examples_section = "\n\n".join(example_blocks)

    task_lines = []
    for idx, item in enumerate(titles_to_classify, start=1):
        scopus_id = item.get("scopus_id", "MISSING_ID")
        title = item["title"]
        task_lines.append(f"{idx}. {scopus_id} — \"{title}\"")
    task_section = "\n".join(task_lines)

    prompt = (
        f"### Song under consideration\n"
        f"{song_title}\n\n"
        f"### Annotated examples\n"
        f"{examples_section}\n\n"
        f"---\n"
        f"### Task\n"
        f"Classify each title below as a *humorous word-play* on the song **{song_title}** "
        f"(`Yes` or `No`) and give a one-sentence explanation.\n"
        f"Return your answers as the JSON array specified in the system instructions.\n\n"
        f"{task_section}"
    )

    return prompt

In [ ]:
def build_system_prompt(song: str) -> str:
    """System instructions defining the wordplay criterion and JSON output schema."""
    return f"""
You are a helpful and witty research assistant specialised in language- and humour-recognition.

## Task
For every paper title you receive, decide whether it is a **humorous word-play** on the Beatles song **“{song}.”**
A title usually qualifies if it…
• Quotes the song verbatim **with one or two words swapped** (e.g. “All You Need Is Data”).  
• Uses a rhyme, pun, or near-synonym in place of the original word(s).  
Exact reuse *without* any twist, or mere topical overlap, is **not** enough.

## Output
Return one JSON array and nothing else.

```json
{{ "scopus_id": "...", "paper_title": "...",
   "explanation": "One sentence.", "answer": "Yes" | "No" }}
```
    """

In [ ]:
def parse_truncated_json(json_text: str):
    """Parse a JSON array, backing off line by line if the response was truncated."""
    # Try parsing, and back off until successful
    lines = json_text.strip().splitlines()
    
    for i in range(len(lines), 0, -1):
        try:
            partial = "\n".join(lines[:i])
            # Try to close the JSON array if it's missing
            if not partial.strip().endswith(']'):
                partial += "\n]"
            return json.loads(partial)
        except json.JSONDecodeError:
            continue

    raise ValueError("Could not parse any valid portion of the JSON.")

In [ ]:
fin = "data/scopus_approx_refs.txt"
fout = "data/scopus_wordplay_tags.txt"

with open(fout, "w", encoding="utf-8") as f:
    f.write("scopus_id\texplanation\twordplay\n")

df = pd.read_csv(fin, sep="\t", encoding="utf-8")
print(df)

In [ ]:
tok_in, tok_out = 0, 0
for song, group in tqdm(df.groupby("song_name"), total=df["song_name"].nunique(), desc="Songs"):
    titles = group[["scopus_id", "title"]].to_dict(orient="records")
    system_prompt = build_system_prompt(song)
    user_prompt = build_user_prompt(song, titles)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=TEMPERATURE,
    )
    tok_in += response.usage.prompt_tokens
    tok_out += response.usage.completion_tokens

    raw_json = response.choices[0].message.content
    clean_json = raw_json.strip().removeprefix("```json").removesuffix("```").strip()
    results = parse_truncated_json(clean_json)

    with open(fout, "a", encoding="utf-8") as f_out:
        for entry in results:
            f_out.write(
                f"{entry['scopus_id']}\t{entry['explanation']}\t{entry['answer']}\n"
            )

In [ ]:
cost = token_costs[MODEL]
in_cost = tok_in / 1_000_000 * cost["input"]
out_cost = tok_out / 1_000_000 * cost["output"]
print(f"Input tokens: {tok_in}, Output tokens: {tok_out}")
print(f"Input cost: ${in_cost:.2f}, output cost: ${out_cost:.2f}")
print(f"Total cost: ${in_cost + out_cost:.2f}")

---
## Inspect results

Join the `Yes` tags back to the candidate titles and report the share classified as
wordplay.

In [ ]:
df_tags = pd.read_csv(fout, sep="\t")
df_refs = pd.read_csv(fin, sep="\t")

# GPT-4o-mini returns scopus_id without the "SCOPUS_ID:" prefix used in the retrieved
# data, so normalise both sides before joining to keep the merge robust to the prefix.
for _df in (df_tags, df_refs):
    _df["scopus_id"] = _df["scopus_id"].astype(str).str.replace("SCOPUS_ID:", "", regex=False)

df_wordplay = pd.merge(
    df_tags[df_tags["wordplay"] == "Yes"], df_refs, on="scopus_id", how="inner"
)

share = len(df_wordplay) / len(df_tags)
print(f"{share:.2%} of the titles were classified as humorous wordplay on a Beatles song title.")
df_wordplay[["song_name", "title", "explanation"]].head(10)